# Extended Analysis: Additional Similarity Metrics

Adds Frechet distance, MSE, and sMAPE to the Pearson/DTW/Cosine metrics from `analysis.ipynb`.
Saves all results to `metrics_results.pkl` for use in `analysis_comparison.ipynb`.

## Setup

Install dependencies and import libraries. `similaritymeasures` provides the discrete Frechet distance implementation.

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'scipy', 'dtw-python', 'similaritymeasures']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy import stats as scipy_stats
from scipy.spatial.distance import cosine as cosine_distance
from dtw import dtw as dtw_func
from matplotlib.lines import Line2D
import similaritymeasures

%matplotlib inline

## Data Loading & Normalization

Load the 4 clusters of gene expression data (~17,856 genes x 4 timepoints each),
apply min-max normalization to (0, 1], and define the constraint patterns that
each metric will match genes against.

In [ ]:
# --- Load data (same as Phase 1 in analysis.ipynb) ---
genes = pd.read_csv("analysis data/gene_list.csv", header=None).squeeze().tolist()

constraints = {
    "clusterOne": [1.45, 12.0, 5.21, 6.51],
    "clusterTwo": [0.371, 0.803, 1.22, 1.78],
    "clusterThree": [0.0405, 0, 0.0369, 0.0353],
    "clusterFour": [0.00913, 0.638, 0.0178, 0.0243]
}
constraintsLT = {
    "clusterOneLT": [.894475, 2.568650, 1.826, 2.016452],
    "clusterTwoLT": [0.036407, 0.589570, 0.798347, 1.023510],
    "clusterThreeLT": [0.039660, 0.0, 0.036266, 0.034714],
    "clusterFourLT": [0.009090, 0.493705, 0.017601, 0.024007]
}
constraint_patterns = {k: {"constraint": v} for k, v in constraints.items()}
constraint_patternsLT = {k: {"constraint": v} for k, v in constraintsLT.items()}

def load_gene_counts(lt=False):
    suffix = "LT" if lt else ""
    folder = f"analysis data/gene_counts{suffix}/"
    cc = {}
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith(".csv"): continue
        df = pd.read_csv(os.path.join(folder, fn), index_col=0)
        df.columns = [0, 3, 6, 9]
        cc[fn.split("_")[0]] = df
    return cc

def normalize_01(cluster_counts):
    eps = 1e-10
    out = {}
    for cn, df in cluster_counts.items():
        rmin, rmax = df.min(axis=1), df.max(axis=1)
        rr = rmax - rmin
        ndf = df.sub(rmin, axis=0).add(eps).div(rr.add(eps), axis=0)
        ndf.loc[rr == 0] = 1.0
        out[cn] = ndf
    return out

def normalize_pattern(pv):
    eps = 1e-10
    p = np.array(pv, dtype=float)
    return (p - p.min() + eps) / (p.max() - p.min() + eps)

gene_counts = load_gene_counts(False)
gene_countsLT = load_gene_counts(True)
gene_counts_norm = normalize_01(gene_counts)
gene_countsLT_norm = normalize_01(gene_countsLT)

for name, df in gene_counts.items():
    print(f"{name}: {df.shape[0]} genes x {df.shape[1]} timepoints")

## Phase 2 (rerun): Full rankings for Pearson, DTW, Cosine

Need all-gene rankings (not just top-20) for ensemble aggregation later.

These are the same Pearson/DTW/Cosine functions from `analysis.ipynb`, modified to
return scores for **all** genes (not just top 20) so the ensemble ranking in
`analysis_comparison.ipynb` can aggregate across the full gene list.

In [ ]:
def all_genes_pearson(patterns, gene_counts_norm):
    """Return full Pearson r for every gene (not just top N)."""
    results = {}
    for cluster_name, cluster_patterns in patterns.items():
        df = gene_counts_norm[cluster_name]
        gene_matrix = df.values
        gene_names = df.index.tolist()
        cluster_results = {}
        for pattern_name, pattern_vec in cluster_patterns.items():
            pattern_arr = np.array(pattern_vec, dtype=float)
            if np.std(pattern_arr) == 0:
                r_values = np.zeros(len(gene_names))
            else:
                r_values = np.empty(len(gene_names))
                for i in range(len(gene_names)):
                    gene_vec = gene_matrix[i]
                    if np.std(gene_vec) == 0:
                        r_values[i] = 0.0
                    else:
                        r, _ = scipy_stats.pearsonr(pattern_arr, gene_vec)
                        r_values[i] = r if np.isfinite(r) else 0.0
            cluster_results[pattern_name] = pd.Series(r_values, index=gene_names)
        results[cluster_name] = cluster_results
    return results

def all_genes_dtw(patterns, gene_counts_norm):
    """Return full DTW distance for every gene."""
    results = {}
    eps = 1e-10
    for cluster_name, cluster_patterns in patterns.items():
        df = gene_counts_norm[cluster_name]
        gene_matrix = df.values
        cluster_results = {}
        for pattern_name, pattern_vals in cluster_patterns.items():
            p = np.array(pattern_vals, dtype=float)
            p_norm = (p - p.min() + eps) / (p.max() - p.min() + eps)
            distances = np.empty(len(df), dtype=float)
            for i in range(len(df)):
                alignment = dtw_func(p_norm, gene_matrix[i].astype(float))
                distances[i] = alignment.distance
            cluster_results[pattern_name] = pd.Series(distances, index=df.index)
        results[cluster_name] = cluster_results
    return results

def all_genes_cosine(patterns, gene_counts_norm):
    """Return full cosine similarity for every gene."""
    results = {}
    for cluster_name, cluster_patterns in patterns.items():
        df = gene_counts_norm[cluster_name]
        gene_matrix = df.values
        gene_names = df.index.tolist()
        cluster_results = {}
        for pattern_name, pattern_vec in cluster_patterns.items():
            pattern_arr = np.array(pattern_vec, dtype=float)
            if np.linalg.norm(pattern_arr) == 0:
                similarities = np.zeros(len(gene_names))
            else:
                similarities = np.empty(len(gene_names), dtype=float)
                for i, gene_vec in enumerate(gene_matrix):
                    if np.linalg.norm(gene_vec) == 0:
                        similarities[i] = 0.0
                    else:
                        similarities[i] = 1.0 - cosine_distance(pattern_arr, gene_vec)
            cluster_results[pattern_name] = pd.Series(similarities, index=gene_names)
        results[cluster_name] = cluster_results
    return results

print("Phase 2 functions defined.")

Run all three Phase 2 metrics on both raw and log-transformed data.
DTW is the slowest (~18s total) since it computes pairwise alignment for each gene.

In [ ]:
# Run Phase 2 metrics (full rankings)
print("Computing Pearson correlation (all genes)...")
pearson_all = all_genes_pearson(constraint_patterns, gene_counts_norm)
pearson_allLT = all_genes_pearson(constraint_patternsLT, gene_countsLT_norm)

print("Computing DTW distance (all genes)... (this takes a few minutes)")
dtw_all = all_genes_dtw(constraint_patterns, gene_counts_norm)
dtw_allLT = all_genes_dtw(constraint_patternsLT, gene_countsLT_norm)

print("Computing Cosine similarity (all genes)...")
cosine_all = all_genes_cosine(constraint_patterns, gene_counts_norm)
cosine_allLT = all_genes_cosine(constraint_patternsLT, gene_countsLT_norm)

print("Phase 2 metrics computed.")

## Phase 5A: Frechet Distance

Measures max pointwise curve distance while preserving temporal ordering (no warping like DTW).
With 4 timepoints this is close to max absolute error — we compare below.

In [ ]:
def all_genes_frechet(patterns, gene_counts_norm):
    """Compute discrete FrÃ©chet distance between pattern and each gene.
    
    Uses similaritymeasures.frechet_dist which expects 2D arrays
    of shape (n_points, n_dims). We reshape our 1D time series as
    (4, 2) curves with x=timepoint, y=value.
    
    Lower distance = more similar.
    """
    x = np.array([0, 3, 6, 9], dtype=float)
    results = {}
    for cluster_name, cluster_patterns in patterns.items():
        print(f"  [FrÃ©chet] Processing {cluster_name}")
        df = gene_counts_norm[cluster_name]
        gene_matrix = df.values
        cluster_results = {}
        for pattern_name, pattern_vals in cluster_patterns.items():
            p_norm = normalize_pattern(pattern_vals)
            # Shape as 2D curve: (timepoint, value)
            pattern_curve = np.column_stack([x, p_norm])
            
            distances = np.empty(len(df), dtype=float)
            for i in range(len(df)):
                gene_curve = np.column_stack([x, gene_matrix[i]])
                distances[i] = similaritymeasures.frechet_dist(pattern_curve, gene_curve)
            
            cluster_results[pattern_name] = pd.Series(distances, index=df.index)
            top = cluster_results[pattern_name].nsmallest(5)
            print(f"    {pattern_name}: top gene = {top.index[0]} (dist={top.iloc[0]:.4f})")
        results[cluster_name] = cluster_results
    return results

print("Computing FrÃ©chet distance (all genes)...")
frechet_all = all_genes_frechet(constraint_patterns, gene_counts_norm)
frechet_allLT = all_genes_frechet(constraint_patternsLT, gene_countsLT_norm)
print("Done.")

## Phase 5B: MSE

Vectorized — runs in <0.1s. Penalizes large deviations more than small ones.

In [ ]:
def all_genes_mse(patterns, gene_counts_norm):
    """Compute MSE between normalized pattern and each gene.
    
    Lower MSE = more similar.
    """
    results = {}
    for cluster_name, cluster_patterns in patterns.items():
        print(f"  [MSE] Processing {cluster_name}")
        df = gene_counts_norm[cluster_name]
        gene_matrix = df.values
        cluster_results = {}
        for pattern_name, pattern_vals in cluster_patterns.items():
            p_norm = normalize_pattern(pattern_vals)
            # Vectorized: MSE for each gene row vs pattern
            diff = gene_matrix - p_norm[np.newaxis, :]
            mse_values = np.mean(diff ** 2, axis=1)
            
            cluster_results[pattern_name] = pd.Series(mse_values, index=df.index)
            top = cluster_results[pattern_name].nsmallest(5)
            print(f"    {pattern_name}: top gene = {top.index[0]} (MSE={top.iloc[0]:.6f})")
        results[cluster_name] = cluster_results
    return results

print("Computing MSE (all genes)...")
mse_all = all_genes_mse(constraint_patterns, gene_counts_norm)
mse_allLT = all_genes_mse(constraint_patternsLT, gene_countsLT_norm)
print("Done.")

## Phase 5C: sMAPE

Symmetric MAPE avoids division-by-zero but has a subtler problem: when the
pattern is near zero, sMAPE saturates at ~2.0 regardless of the gene's value,
losing all discriminating power at that timepoint. Watch for this in clusterThree.

In [ ]:
def all_genes_smape(patterns, gene_counts_norm):
    """Compute symmetric MAPE between normalized pattern and each gene.
    
    sMAPE = mean(2 * |p - g| / (|p| + |g| + eps)) across timepoints.
    Range: [0, 2]. Lower = more similar.
    
    Uses symmetric form to handle zero values in patterns (e.g., clusterThree at t=3).
    """
    eps = 1e-10
    results = {}
    for cluster_name, cluster_patterns in patterns.items():
        print(f"  [sMAPE] Processing {cluster_name}")
        df = gene_counts_norm[cluster_name]
        gene_matrix = df.values
        cluster_results = {}
        for pattern_name, pattern_vals in cluster_patterns.items():
            p_norm = normalize_pattern(pattern_vals)
            # Vectorized sMAPE
            abs_diff = np.abs(gene_matrix - p_norm[np.newaxis, :])
            denom = np.abs(gene_matrix) + np.abs(p_norm[np.newaxis, :]) + eps
            smape_values = np.mean(2.0 * abs_diff / denom, axis=1)
            
            cluster_results[pattern_name] = pd.Series(smape_values, index=df.index)
            top = cluster_results[pattern_name].nsmallest(5)
            print(f"    {pattern_name}: top gene = {top.index[0]} (sMAPE={top.iloc[0]:.4f})")
        results[cluster_name] = cluster_results
    return results

print("Computing sMAPE (all genes)...")
smape_all = all_genes_smape(constraint_patterns, gene_counts_norm)
smape_allLT = all_genes_smape(constraint_patternsLT, gene_countsLT_norm)
print("Done.")

## Phase 5 Validation: 6-method overlap matrices

Key questions: Do Frechet/MSE cluster with existing methods? Does sMAPE diverge?

In [ ]:
def get_top_n(scores, cluster, pattern, n=20, ascending=True):
    """Get top N genes. ascending=True for distance metrics (lower=better),
    ascending=False for similarity metrics (higher=better)."""
    s = scores[cluster][pattern]
    if ascending:
        return set(s.nsmallest(n).index)
    else:
        return set(s.nlargest(n).index)

# All methods with their sorting direction
all_methods = {
    "Pearson": (pearson_all, False),    # higher = better
    "DTW": (dtw_all, True),             # lower = better
    "Cosine": (cosine_all, False),      # higher = better
    "FrÃ©chet": (frechet_all, True),     # lower = better
    "MSE": (mse_all, True),             # lower = better
    "sMAPE": (smape_all, True),         # lower = better
}

all_methodsLT = {
    "Pearson": (pearson_allLT, False),
    "DTW": (dtw_allLT, True),
    "Cosine": (cosine_allLT, False),
    "FrÃ©chet": (frechet_allLT, True),
    "MSE": (mse_allLT, True),
    "sMAPE": (smape_allLT, True),
}

method_names = list(all_methods.keys())
n = 20

print("Phase 5 Validation: Top-20 overlap matrix (raw data)")
print("=" * 70)

for cluster_name in constraint_patterns:
    print(f"\n--- {cluster_name} ---")
    
    # Get top-20 gene sets for each method
    top_sets = {}
    for mname, (scores, asc) in all_methods.items():
        top_sets[mname] = get_top_n(scores, cluster_name, "constraint", n=n, ascending=asc)
    
    # Print overlap matrix
    header = f"{'':>10}" + "".join(f"{m:>10}" for m in method_names)
    print(header)
    for m1 in method_names:
        row = f"{m1:>10}"
        for m2 in method_names:
            overlap = len(top_sets[m1] & top_sets[m2])
            row += f"{overlap:>10}"
        print(row)

In [ ]:
# Visual: heatmap of all 6 methods for each cluster
nm = len(method_names)

for label, methods_dict, pat_dict in [
    ("Raw", all_methods, constraint_patterns),
    ("LT", all_methodsLT, constraint_patternsLT)
]:
    for cluster_name in pat_dict:
        top_sets = {}
        for mname, (scores, asc) in methods_dict.items():
            top_sets[mname] = get_top_n(scores, cluster_name, "constraint", n=n, ascending=asc)
        
        overlap_matrix = np.zeros((nm, nm), dtype=int)
        for i, m1 in enumerate(method_names):
            for j, m2 in enumerate(method_names):
                overlap_matrix[i, j] = len(top_sets[m1] & top_sets[m2])
        
        fig, ax = plt.subplots(figsize=(7, 6))
        im = ax.imshow(overlap_matrix, cmap="YlOrRd", vmin=0, vmax=n)
        ax.set_xticks(range(nm))
        ax.set_yticks(range(nm))
        ax.set_xticklabels(method_names, rotation=45, ha="right")
        ax.set_yticklabels(method_names)
        ax.set_title(f"Top-{n} Gene Overlap | {cluster_name} ({label})")
        for i in range(nm):
            for j in range(nm):
                ax.text(j, i, str(overlap_matrix[i, j]), ha="center", va="center",
                        fontsize=12, color="white" if overlap_matrix[i, j] > n / 2 else "black")
        plt.colorbar(im, ax=ax, label="Shared genes")
        plt.tight_layout()
        plt.show()

## Save Results

Save all metric scores to `metrics_results.pkl` so `analysis_comparison.ipynb`
can load them instantly without recomputing.

In [ ]:
# Save all metric results for the comparison notebook
import pickle

results = {
    'constraint_patterns': constraint_patterns,
    'constraint_patternsLT': constraint_patternsLT,
    'all_methods': {
        'Pearson': (pearson_all, False), 'DTW': (dtw_all, True), 'Cosine': (cosine_all, False),
        'Frechet': (frechet_all, True), 'MSE': (mse_all, True), 'sMAPE': (smape_all, True),
    },
    'all_methodsLT': {
        'Pearson': (pearson_allLT, False), 'DTW': (dtw_allLT, True), 'Cosine': (cosine_allLT, False),
        'Frechet': (frechet_allLT, True), 'MSE': (mse_allLT, True), 'sMAPE': (smape_allLT, True),
    },
}
pickle.dump(results, open('metrics_results.pkl', 'wb'))
print("Saved metrics_results.pkl")